# **ST 554 Spring 2026 Final Project**
## *Created by Cody Ashby on April 27, 2026*
### Fitting The Model

In [1]:
import pandas as pd
import numpy as np
import time
from pyspark.sql import SparkSession
spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
from pyspark.sql.functions import window, col
from pyspark.sql.types import StructType
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, SQLTransformer, VectorAssembler, Binarizer, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder # set parallelism to 128 when doing CV!
from pyspark.ml.evaluation import RegressionEvaluator

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/29 12:13:01 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
power_data = pd.read_csv("power_ml_data.csv")

In [3]:
spark_power_data = spark.read.csv("power_ml_data.csv",inferSchema=True,header=True)
spark_power_data.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', IntegerType(), True), StructField('Hour', IntegerType(), True)])

In [4]:
sqlTrans=SQLTransformer(statement="SELECT Month, Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, Diffuse_Flows, \
                                        Power_Zone_1, Power_Zone_2, Power_Zone_3 as label, CAST(Hour AS DOUBLE) AS Revised_Hour FROM __THIS__")

In [5]:
binary_Hour=Binarizer(threshold=6.5,inputCol="Revised_Hour",outputCol="Binary_Hour")

In [6]:
#Month_Index=StringIndexer(inputCol="Month",outputCol="Indexed_Month")
Month_OHE=OneHotEncoder(inputCol="Month",outputCol="Encoded_Month")

In [7]:
Weather_Vars=VectorAssembler(inputCols=['Temperature','Humidity','Wind_Speed','General_Diffuse_Flows','Diffuse_Flows'],outputCol='weather_features',handleInvalid='keep')
Weather_Comps=Weather_Vars.transform(spark_power_data)
PC_features=PCA(k=2,inputCol='weather_features',outputCol="Prin_Comps")
Fitted_PC=PC_features.fit(Weather_Comps)

In [8]:
total_features=VectorAssembler(inputCols=['Prin_Comps','Binary_Hour','Power_Zone_1','Power_Zone_2','Encoded_Month'],outputCol='features',handleInvalid='keep')

In [9]:
MLR=LinearRegression()
grand_pipeline=Pipeline(stages=[sqlTrans,binary_Hour,Month_OHE,Weather_Vars,PC_features,total_features,MLR])

In [10]:
EN_Param_Grid=ParamGridBuilder().addGrid(MLR.regParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]) \
                            .addGrid(MLR.elasticNetParam,[0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.98,0.99,1]).build()

In [13]:
ElasticNet_CrossVal=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=EN_Param_Grid,
                                   evaluator=RegressionEvaluator(metricName='rmse'),
                                   numFolds=5,
                                   seed=8,
                                   parallelism=128)
ElasticNet_Model=ElasticNet_CrossVal.fit(spark_power_data)

In [15]:
ElasticNet_RMSEs=[]
for _ in range(len(EN_Param_Grid)):
    ElasticNet_RMSEs.append([ElasticNet_Model.avgMetrics[_],EN_Param_Grid[_].values()])
ElasticNet_RMSEs

[[2147.743822079609, dict_values([0.0, 0.0])],
 [2147.743822079609, dict_values([0.0, 0.05])],
 [2147.743822079609, dict_values([0.0, 0.1])],
 [2147.743822079609, dict_values([0.0, 0.25])],
 [2147.743822079609, dict_values([0.0, 0.5])],
 [2147.743822079609, dict_values([0.0, 0.75])],
 [2147.743822079609, dict_values([0.0, 0.9])],
 [2147.743822079609, dict_values([0.0, 0.95])],
 [2147.743822079609, dict_values([0.0, 0.98])],
 [2147.743822079609, dict_values([0.0, 0.99])],
 [2147.743822079609, dict_values([0.0, 1.0])],
 [2147.743804871119, dict_values([0.05, 0.0])],
 [2147.745520060856, dict_values([0.05, 0.05])],
 [2147.738868265524, dict_values([0.05, 0.1])],
 [2147.7435245465463, dict_values([0.05, 0.25])],
 [2147.743723303225, dict_values([0.05, 0.5])],
 [2147.7424631014997, dict_values([0.05, 0.75])],
 [2147.7427017257346, dict_values([0.05, 0.9])],
 [2147.74267120382, dict_values([0.05, 0.95])],
 [2147.742633940907, dict_values([0.05, 0.98])],
 [2147.742635289277, dict_values([0.05

Upon close inspection of each RMSE associated with each pair of hyperparameters, we can see that the lowest RMSE goes to the one with the `regParam` of 0.05 and the `ElasticNetParam` of 0.1, indicating fairly minimal regularization required for this model. The CV error, as well as the training RMSE, for this model is shown below.

In [16]:
ElasticNet_CV_Error=np.mean(ElasticNet_Model.avgMetrics)
ElasticNet_Training_RMSE=min(ElasticNet_Model.avgMetrics)
print(ElasticNet_CV_Error)
print(ElasticNet_Training_RMSE)

2147.7574775784437
2147.738868265524


With those numbers in mind, we will now build our optimal Elastic Net model using the tuning parameters associated with the lowest RMSE.

In [17]:
Best_EN_Params=ParamGridBuilder().addGrid(MLR.regParam,[0.05]).addGrid(MLR.elasticNetParam,[0.1]).build()
Best_ElasticNet_CV=CrossValidator(estimator=grand_pipeline,
                                   estimatorParamMaps=Best_EN_Params,
                                   evaluator=RegressionEvaluator(labelCol='label',metricName='rmse'),
                                   numFolds=5)
Best_ElasticNet_Model=Best_ElasticNet_CV.fit(spark_power_data)

This fitted model will now be used to make predictions based on the original dataset. We'll also note how far each prediction is from the actual response with a `residuals` column, as shown below.

In [18]:
Best_ElasticNet_Model_Preds=Best_ElasticNet_Model.transform(spark_power_data)
Best_ElasticNet_Model_Preds_Resids=Best_ElasticNet_Model_Preds.withColumn("residuals",col("label")-col("prediction"))
Best_ElasticNet_Model_Preds_Resids.select("label","prediction","residuals").show()

+-----------+------------------+------------------+
|      label|        prediction|         residuals|
+-----------+------------------+------------------+
|20240.96386| 20878.85066079531|-637.8868007953097|
|20131.08434|18660.227266546775| 1470.857073453226|
|19668.43373| 18204.75215311686|1463.6815768831402|
|18899.27711|17590.648498340925|1308.6286116590745|
|18442.40964|  16997.3019864581|1445.1076535419015|
|18130.12048|16517.686723495084|1612.4337565049173|
|17945.06024| 16093.24614105391| 1851.814098946088|
|17459.27711|15722.695360254002|1736.5817497459975|
|17025.54217|15271.043828662507| 1754.498341337494|
|16794.21687|14938.348764665025|1855.8681053349756|
|16638.07229|14652.383721423037| 1985.688568576963|
|16395.18072|14414.900662729378| 1980.280057270622|
|16117.59036|14082.889275685437|2034.7010843145636|
| 15822.6506|13624.882091434294|2197.7685085657067|
|15672.28916|13450.340775466415|2221.9483845335853|
|15597.10843| 13302.24639455996| 2294.862035440041|
|15510.36145

In [ ]:
#The overall idea of Best_ElasticNet_Model_Preds_Resids will be saved as a dataframe for later use. Namely, when we use incoming data to make predictions.

### Streaming Data

In [20]:
power_streaming_data=pd.read_csv("Other_Files/power_streaming_data.csv")

In [21]:
spark_power_streaming_data=spark.read.csv("Other_Files/power_streaming_data.csv",inferSchema=True,header=True)
spark_power_streaming_data.schema

StructType([StructField('Temperature', DoubleType(), True), StructField('Humidity', DoubleType(), True), StructField('Wind_Speed', DoubleType(), True), StructField('General_Diffuse_Flows', DoubleType(), True), StructField('Diffuse_Flows', DoubleType(), True), StructField('Power_Zone_1', DoubleType(), True), StructField('Power_Zone_2', DoubleType(), True), StructField('Power_Zone_3', DoubleType(), True), StructField('Month', IntegerType(), True), StructField('Hour', IntegerType(), True)])

In [22]:
Read_Power_Stream=spark.readStream.schema(spark_power_data.schema).csv("Other_Files",header=True)

In [33]:
Streamed_Preds=Best_ElasticNet_Model.transform(Read_Power_Stream)
Streamed_Preds_Renamed=Read_Power_Stream.withColumnRenamed("Power_Zone_3","label") \
                                        .select("label","Month","Temperature","Humidity","Wind_Speed","General_Diffuse_Flows", \
                                                "Diffuse_Flows","Power_Zone_1","Power_Zone_2","Month","Hour")
Streamed_Preds_Resids=Streamed_Preds.withColumn("residual",col("label")-col("prediction")).select("label","prediction","residual")
Joined_Streams=Streamed_Preds_Resids.join(Streamed_Preds_Renamed,"label","inner")

In [31]:
Power_Query=Streamed_Preds_Renamed.writeStream.outputMode("append").format("console").start()

-------------------------------------------
Batch: 0
-------------------------------------------
+-----+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+------------+-----------+--------------+--------------------+--------------------+--------------------+------------------+
|Month|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Revised_Hour|Binary_Hour| Encoded_Month|    weather_features|          Prin_Comps|            features|        prediction|
+-----+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+------------+-----------+--------------+--------------------+--------------------+--------------------+------------------+
|    1|      4.805|    76.2|     0.081|                0.059|        0.134| 20421.26582| 12908.20669|14590.84337|         3.0|        0.0|(12,[1],[1.0])|[4.805,76.2,0.081...|[1.86208914981191...|

### Producing Data

Now that the query has started, let's import the file that allows us to view the output.

In [ ]:
%run Other_Files/PowerStreamDataProducer.py


-------------------------------------------
Batch: 1
-------------------------------------------
-------------------------------------------
Batch: 1
-------------------------------------------
+-----------+----------+--------+
|      label|prediction|residual|
+-----------+----------+--------+
|21369.04277|       NaN|     NaN|
|19407.90274|       NaN|     NaN|
|26956.73519|       NaN|     NaN|
|28150.20747|       NaN|     NaN|
|32601.26582|       NaN|     NaN|
+-----------+----------+--------+

+-----+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+------------+-----------+-------------+--------------------+--------------------+--------------------+----------+
|Month|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Revised_Hour|Binary_Hour|Encoded_Month|    weather_features|          Prin_Comps|            features|prediction|
+-----+-----------+--------+----------+------

In [ ]:
#Power_Query.stop() will be used to stop the query (if need be).